# makemore Part 1 — positional encoding + 2-char history (a linear 4-gram)

Extends the neural bigram model from Andrej Karpathy's [makemore Part 1](https://www.youtube.com/watch?v=TCH_1BHY58I) in two ways:

1. **Positional encoding** — append a one-hot of each character's position *within its word*.
2. **2 characters of history** — feed the current character plus the previous two, so the
   model sees a 3-character context (effectively a linear 4-gram).

Input layout (**104 dims**): `[ char_{t-2} (27) | char_{t-1} (27) | char_t (27) | position (23) ]`,
so the weight matrix `W` is **104 x 27**. Everything else (softmax, NLL loss, gradient descent)
is identical to the original.

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
print('num words:', len(words))
print('first few:', words[:8])

num words: 32033
first few: ['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [3]:
# build the character <-> integer mappings ('.' is the start/end boundary token -> 0)
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [4]:
# create the bigram dataset: xs = current char, ys = next char
xs, ys = [], []
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        xs.append(stoi[ch1])
        ys.append(stoi[ch2])
xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print('number of examples:', num)

number of examples: 228146


## Position within the word

A new word begins wherever `xs == 0` (the `.` boundary token). We want a counter that resets to 0
at every boundary and increments by 1 for each following character. Done vectorized via `cummax`:
`boundary_idx` holds each row's index at word starts (0 elsewhere); a running max propagates the
index of the most recent word start, and subtracting it from the running index gives the position.

In [5]:
def positions_of(xs):
    idx = torch.arange(xs.shape[0], device=xs.device)
    boundary_idx = torch.where(xs == 0, idx, torch.zeros_like(idx))
    last_boundary = torch.cummax(boundary_idx, dim=0).values
    return idx - last_boundary   # 0 at each word start, then 1, 2, 3, ...

# sanity check on 'emma' (xs = . e m m a) then 'ava' (. a v a)
demo = torch.tensor([0, 5, 13, 13, 1, 0, 1, 22, 1])
print('xs:       ', demo.tolist())
print('positions:', positions_of(demo).tolist())

xs:        [0, 5, 13, 13, 1, 0, 1, 22, 1]
positions: [0, 1, 2, 3, 4, 0, 1, 2, 3]


## Encoding: 2-char history + position

For each example we build three character one-hots — `char_{t-2}`, `char_{t-1}`, `char_t` — and the
position one-hot. The history slots are masked to **zero when they would cross a word boundary**
(at position 0 there is no `t-1`; at position 1 there is no `t-2`). The window naturally *shifts*
forward because `char_{t-1}` / `char_{t-2}` are just `xs` shifted by 1 and 2.

In [6]:
def encode_xs(xs, max_chars=23):
    pos = positions_of(xs)

    # current char and the two previous chars (shifted views of xs)
    e_t = F.one_hot(xs, num_classes=27).float()
    xs1 = torch.cat([torch.zeros(1, dtype=xs.dtype), xs[:-1]]); e1 = F.one_hot(xs1, 27).float()
    xs2 = torch.cat([torch.zeros(2, dtype=xs.dtype), xs[:-2]]); e2 = F.one_hot(xs2, 27).float()

    # zero out history that crosses a word boundary
    e1[pos < 1] = 0.0   # no t-1 at the first char of a word
    e2[pos < 2] = 0.0   # no t-2 in the first two chars of a word

    # one-hot position within the word
    penc = F.one_hot(pos.clamp(max=max_chars - 1), num_classes=max_chars).float()

    # [ char_{t-2} | char_{t-1} | char_t | position ]  -> (N, 104)
    return torch.cat([e2, e1, e_t, penc], dim=1)

xenc = encode_xs(xs)
print('encoded input shape:', xenc.shape)   # (num, 104)

encoded input shape: torch.Size([228146, 104])


## Train

Same training loop as the original (softmax -> negative log-likelihood + small L2 -> gradient
descent). The only knob changed is the learning rate: because the input now carries three
one-hots (it sums to ~4 instead of ~2), the original `lr=50` overshoots, so we use `lr=25`.

In [7]:
# initialize the weights:  104 inputs -> 27 outputs
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((104, 27), generator=g, requires_grad=True)

xenc = encode_xs(xs)   # constant across steps, so compute it once

for k in range(500):
    # forward pass
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdims=True)
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01 * (W**2).mean()
    if k % 50 == 0 or k == 499:
        print(k, loss.item())

    # backward pass
    W.grad = None
    loss.backward()

    # update
    W.data += -25 * W.grad

print('final loss:', loss.item())

0 4.55971097946167


50 2.352071762084961


100 2.2823333740234375


150 2.2577826976776123


200 2.2451913356781006


...

300 2.2328755855560303


350 2.22951340675354


400 2.2270498275756836


450 2.2251698970794678


499 2.223717212677002
final loss: 2.223717212677002


For reference: the plain bigram baseline converges to **~2.483**, the bigram + position model to
**~2.317**, and this 2-char-history + position model to **~2.22** — about an 11% reduction in loss
over the original bigram, using nothing but a richer input to a single linear layer.

## A reusable word generator (with optional history)

`generate_word` mirrors the training encoding exactly: it keeps a sliding window of the previously
emitted characters, builds the same `[char_{t-2} | char_{t-1} | char_t | position]` input, samples
the next character, then **shifts** the window. Passing `use_history=False` zeroes the two history
slots.

> **Note:** `use_history=False` is an *ablation* of this history-trained model — we feed it inputs
> it never saw in training (history always zero), which is out-of-distribution. For a *fair*
> no-history baseline we train a separate position-only model further below.

In [8]:
def generate_word(W, generator, use_history=True, max_chars=23):
    """Sample one word. If use_history=False the two history slots are zeroed,
    so the same trained model predicts from only the current char + position."""
    out = []          # emitted characters
    hist = []         # previously emitted char ids within this word
    ix = 0            # current char id, starts at '.' (the boundary token)
    while True:
        pos = min(len(out), max_chars - 1)
        e_t = F.one_hot(torch.tensor([ix]), num_classes=27).float()
        e1 = torch.zeros((1, 27)); e2 = torch.zeros((1, 27))
        if use_history:
            if len(hist) >= 1: e1 = F.one_hot(torch.tensor([hist[-1]]), 27).float()
            if len(hist) >= 2: e2 = F.one_hot(torch.tensor([hist[-2]]), 27).float()
        penc = F.one_hot(torch.tensor([pos]), num_classes=max_chars).float()
        xenc = torch.cat([e2, e1, e_t, penc], dim=1)            # (1, 104)

        logits = xenc @ W
        p = logits.exp(); p = p / p.sum(1, keepdims=True)
        nix = torch.multinomial(p, num_samples=1, replacement=True, generator=generator).item()

        out.append(itos[nix])
        hist.append(ix)        # shift: the current char becomes history
        ix = nix
        if ix == 0:            # emitted the end-of-word '.'
            break
    return ''.join(out)

def generate(n=10, seed=2147483647, use_history=True):
    """Generate n words from the trained model."""
    g = torch.Generator().manual_seed(seed)
    return [generate_word(W, g, use_history=use_history) for _ in range(n)]

In [9]:
# 5 words WITH history vs 5 words WITHOUT history (same model, same seed)
print('with history     :', generate(5, use_history=True))
print('without history  :', generate(5, use_history=False))

with history     : ['cexbe.', 'morlyur.', 'rochitah.', 'mellima.', 'tainalla.']
without history  : ['cexbe.', 'momonura.', 'cezitynnevizimigtammmbgghn.', 'kavia.', 'semivie.']


## A fair no-history baseline: position-only model

The ablation above is misleading: zeroing history on a *history-trained* model feeds it
out-of-distribution inputs, so it degenerates (run-on names). The honest comparison is a model
that is **actually trained** with position but no history — a 50-dim `[char_t | position]` input,
exactly the model from the positional notebook (loss ~2.317). A model trained this way learns
weights appropriate for having no history, so it stays coherent.

In [10]:
# position-only encoder: current char + position, NO history  -> 50 dims
def encode_pos(xs, max_chars=23):
    pos = positions_of(xs)
    e_t = F.one_hot(xs, num_classes=27).float()
    penc = F.one_hot(pos.clamp(max=max_chars - 1), num_classes=max_chars).float()
    return torch.cat([e_t, penc], dim=1)   # (N, 50)

# train it (same recipe as the original positional notebook: lr=50, 200 steps)
g = torch.Generator().manual_seed(2147483647)
W_pos = torch.randn((50, 27), generator=g, requires_grad=True)
xenc_pos = encode_pos(xs)
for k in range(200):
    logits = xenc_pos @ W_pos
    counts = logits.exp(); probs = counts / counts.sum(1, keepdims=True)
    loss_pos = -probs[torch.arange(num), ys].log().mean() + 0.01 * (W_pos**2).mean()
    W_pos.grad = None; loss_pos.backward(); W_pos.data += -50 * W_pos.grad
print('position-only final loss:', loss_pos.item())

position-only final loss: 2.317017078399658


In [11]:
def generate_word_pos(W, generator, max_chars=23):
    """Sample one word from a position-only model (char + position, no history)."""
    out = []; ix = 0
    while True:
        pos = min(len(out), max_chars - 1)
        e_t = F.one_hot(torch.tensor([ix]), num_classes=27).float()
        penc = F.one_hot(torch.tensor([pos]), num_classes=max_chars).float()
        xenc = torch.cat([e_t, penc], dim=1)                   # (1, 50)
        p = (xenc @ W).exp(); p = p / p.sum(1, keepdims=True)
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=generator).item()
        out.append(itos[ix])
        if ix == 0:
            break
    return ''.join(out)

### Side-by-side: position-only vs position + history

Same sampling seed for both. Both are coherent (no run-ons) — positional encoding alone already
teaches the model when to stop. The history model's names tend to be a touch richer / better
structured (smoother letter clusters), which is the incremental ~0.09 in loss showing up
qualitatively.

In [12]:
gp = torch.Generator().manual_seed(2147483647)
gh = torch.Generator().manual_seed(2147483647)
names_pos  = [generate_word_pos(W_pos, gp) for _ in range(20)]
names_hist = [generate_word(W, gh, use_history=True) for _ in range(20)]

print(f"{'POSITION ONLY (loss~2.32)':<28} | {'POSITION + HISTORY (loss~2.22)':<28}")
print('-' * 60)
for a, b in zip(names_pos, names_hist):
    print(f'{a:<28} | {b:<28}')

POSITION ONLY (loss~2.32)    | POSITION + HISTORY (loss~2.22)
------------------------------------------------------------
cexze.                       | cexbe.                      
moman.                       | morlyur.                    
raile.                       | rochitah.                   
kayha.                       | mellima.                    
konimi.                      | tainalla.                   
tainella.                    | kaman.                      
kaka.                        | arreen.                     
da.                          | puelthar.                   
samiyau.                     | gotti.                      
javer.                       | moriel.                     
gotai.                       | kausif.                     
moliel.                      | teda.                       
kausie.                      | kaley.                      
teda.                        | maside.                     
kaleyna.                     | enkavi